# Step 4.04: Conference comparison for the t=-2 event study analysis

This notebook creates a compact two page comparison figure for ICFP, POPL, and PLDI 2021 onward. The first page shows all retained event units. The second page shows the stricter sample with no earlier service.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from datetime import date
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import IFrame, Markdown, display
from matplotlib import font_manager
from matplotlib.backends.backend_pdf import PdfPages

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

from project_setup import ensure_dirs, setup_project

setup = setup_project()
PROJECT = setup.project_folder

STEP_4_PREPARED = PROJECT / "step_4_data" / "prepared"
STEP_4_SUMMARY = PROJECT / "step_4_artifacts" / "summary_tables"
T_MINUS_EVENT_FIGURES = PROJECT / "step_4_artifacts" / "figures_t_minus_event_study"
REPORT_FIGURES = PROJECT / "report_latex" / "figures"
ensure_dirs(STEP_4_SUMMARY, T_MINUS_EVENT_FIGURES, REPORT_FIGURES)

EVENT_ROWS_IN = STEP_4_PREPARED / "t_minus_2_reference_event_window_rows.parquet"
COMPARISON_FIGURE_OUT = T_MINUS_EVENT_FIGURES / "icfp_popl_pldi_comparison.pdf"
REPORT_COMPARISON_FIGURE_OUT = REPORT_FIGURES / "t_minus_icfp_popl_pldi_comparison.pdf"
SUMMARY_OUT = STEP_4_SUMMARY / "tminus_conf_comparison.csv"

EVENT_TIMES = np.array([-2, -1, 0, 1, 2])
BOOTSTRAP_REPS = 2000

print("Project folder: .")
print(f"Run mode: {setup.run_mode}")
print(f"Run date: {date.today().isoformat()}")

## 2. Plot style

In [ ]:
font_path = PROJECT / "fonts" / "LinLibertine_R.ttf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    font_prop = font_manager.FontProperties(fname=font_path)
    font_family = font_prop.get_name()
else:
    font_family = "serif"

mpl.rcParams.update(
    {
        "axes.titlesize": 10,
        "axes.labelsize": 8.5,
        "font.size": 8.8,
        "legend.fontsize": 8,
        "xtick.labelsize": 7.8,
        "ytick.labelsize": 7.8,
        "font.family": font_family,
        "text.usetex": True,
        "axes.linewidth": 0.75,
        "axes.edgecolor": "black",
        "xtick.direction": "out",
        "ytick.direction": "out",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    }
)

CONFERENCE_COLORS = {
    "ICFP": "#f5bde6",
    "POPL": "#81c8be",
    "PLDI 2021 onward": "#e5c890",
}

## 3. Load event window rows

In [ ]:
event_rows = pd.read_parquet(EVENT_ROWS_IN)
event_rows = event_rows.loc[event_rows["plot_group"].isin(CONFERENCE_COLORS)].copy()
event_rows["event_time"] = event_rows["event_time"].astype(int)

required_columns = [
    "event_unit_id",
    "plot_group",
    "event_time",
    "citation_count",
    "delta_from_t_minus_2_log10_citations",
    "career_age_at_first_pc",
    "h_index",
    "h_index_fetch_date",
    "is_true_first_broad_service_in_observed_year_conf",
]
missing_columns = sorted(set(required_columns) - set(event_rows.columns))
assert not missing_columns, missing_columns

print(f"event rows: {event_rows.shape[0]:,}")
print(f"event units: {event_rows['event_unit_id'].nunique():,}")
display(
    event_rows[
        ["event_unit_id", "name", "plot_group", "event_time", "citation_count"]
    ].head()
)

## 4. Plot helpers

In [ ]:
def bootstrap_summary(frame, value_col, event_times=EVENT_TIMES, seed=1234):
    if frame.empty:
        return pd.DataFrame()

    matrix = (
        frame.pivot_table(
            index="event_unit_id",
            columns="event_time",
            values=value_col,
            aggfunc="mean",
        )
        .reindex(columns=event_times)
        .to_numpy()
    )
    n_units = matrix.shape[0]
    if n_units == 0:
        return pd.DataFrame()

    means = np.nanmean(matrix, axis=0)
    rng = np.random.default_rng(seed)
    boot = np.empty((BOOTSTRAP_REPS, len(event_times)))
    for draw in range(BOOTSTRAP_REPS):
        idx = rng.choice(n_units, size=n_units, replace=True)
        boot[draw] = np.nanmean(matrix[idx], axis=0)

    return pd.DataFrame(
        {
            "event_time": event_times,
            "mean": means,
            "ci_lower": np.nanpercentile(boot, 2.5, axis=0),
            "ci_upper": np.nanpercentile(boot, 97.5, axis=0),
            "n_event_units": n_units,
        }
    )


def event_units(frame):
    return frame.sort_values(["event_unit_id", "event_time"]).drop_duplicates(
        "event_unit_id"
    )


def style_axis(ax):
    ax.grid(axis="y", color="0.88", linestyle=":", linewidth=0.55)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def draw_event_axis(ax, rows, color, title, show_ylabel=True):
    summary = bootstrap_summary(
        rows,
        "delta_from_t_minus_2_log10_citations",
        seed=1234,
    )
    ax.axhline(0, color="0.78", linewidth=0.45, zorder=0)
    ax.axvline(0, color="black", linewidth=0.75, linestyle="--")

    if not summary.empty:
        yerr = np.vstack(
            [
                summary["mean"].to_numpy() - summary["ci_lower"].to_numpy(),
                summary["ci_upper"].to_numpy() - summary["mean"].to_numpy(),
            ]
        )
        ax.scatter(
            summary["event_time"],
            summary["mean"],
            marker="o",
            s=20,
            color=color,
            zorder=3,
        )
        ax.errorbar(
            summary["event_time"],
            summary["mean"],
            yerr=yerr,
            fmt="none",
            ecolor="black",
            elinewidth=0.85,
            capsize=2.5,
            capthick=0.85,
            zorder=4,
        )

    ax.set_title(title, loc="left", pad=4)
    ax.set_xticks(EVENT_TIMES)
    ax.set_xlabel("Event time")
    if show_ylabel:
        ax.set_ylabel(r"$\Delta \log_{10}(\mathrm{citations}+1)$ from $t_{-2}$")
    else:
        ax.set_ylabel("")
    style_axis(ax)


def draw_hist_axis(ax, values, bins, color, title, xlabel, xlim):
    clean = pd.to_numeric(values, errors="coerce").dropna()
    ax.hist(clean, bins=bins, color=color, alpha=0.22, edgecolor="none")
    if len(clean):
        jitter_y = np.random.default_rng(1234).uniform(0.25, 1.25, size=len(clean))
        ax.scatter(clean, jitter_y, s=5.5, color="0.38", alpha=0.45, linewidths=0)
        ax.axvline(clean.median(), color="black", linewidth=0.9)
        ax.axvline(clean.mean(), color=color, linewidth=1.0, linestyle="--")
        title = f"{title}\nmedian={clean.median():.1f}, mean={clean.mean():.1f}"
    ax.set_title(title, loc="left", pad=4)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Event units")
    ax.set_xlim(*xlim)
    style_axis(ax)


def draw_pre_pc_axis(ax, rows, color):
    wide = rows.pivot_table(
        index="event_unit_id",
        columns="event_time",
        values="citation_count",
        aggfunc="mean",
    )
    wide = wide.reindex(columns=[-2, -1])
    plot_rows = wide.dropna(how="all")
    data = [plot_rows[-2].dropna().to_numpy(), plot_rows[-1].dropna().to_numpy()]

    ax.boxplot(
        data,
        positions=[0, 1],
        widths=0.46,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "black", "linewidth": 0.85},
        boxprops={
            "facecolor": color,
            "alpha": 0.20,
            "edgecolor": "black",
            "linewidth": 0.75,
        },
        whiskerprops={"color": "black", "linewidth": 0.75},
        capprops={"color": "black", "linewidth": 0.75},
    )

    rng = np.random.default_rng(1234)
    for _, row in plot_rows.iterrows():
        vals = row[[-2, -1]].to_numpy(dtype=float)
        if np.isfinite(vals).all():
            ax.plot([0, 1], vals, color="0.78", alpha=0.38, linewidth=0.45, zorder=1)
        for xpos, value in zip([0, 1], vals):
            if np.isfinite(value):
                ax.scatter(
                    xpos + rng.uniform(-0.06, 0.06),
                    value,
                    s=5.5,
                    color="0.42",
                    alpha=0.42,
                    linewidths=0,
                    zorder=2,
                )

    means = plot_rows[[-2, -1]].mean()
    ax.plot(
        [0, 1],
        [means[-2], means[-1]],
        color=color,
        marker="o",
        markersize=3.2,
        linewidth=1.1,
    )
    ax.set_title(
        f"Pre-PC raw citations\nmean: $t_{{-2}}$ {means[-2]:.1f}, "
        f"$t_{{-1}}$ {means[-1]:.1f}",
        loc="left",
        pad=4,
    )
    ax.set_xticks([0, 1])
    ax.set_xticklabels([r"$t_{-2}$", r"$t_{-1}$"])
    ax.set_xlabel("Pre-PC event time")
    ax.set_ylabel("Citations")
    style_axis(ax)


def broad_first_service_mask(frame):
    return frame["is_true_first_broad_service_in_observed_year_conf"].eq(True)

## 5. Build the two page comparison PDF

In [ ]:
plot_groups = ["ICFP", "POPL", "PLDI 2021 onward"]

unit_rows = event_units(event_rows)
career_age_upper = int(np.ceil(unit_rows["career_age_at_first_pc"].max() / 5) * 5)
career_age_upper = max(10, career_age_upper)
career_age_bins = list(range(0, career_age_upper + 5, 5))
h_index_upper = int(np.ceil(unit_rows["h_index"].max() / 10) * 10)
h_index_upper = max(10, h_index_upper)
h_index_bins = list(range(0, h_index_upper + 10, 10))
h_index_dates = unit_rows["h_index_fetch_date"].dropna().astype(str).sort_values().unique()
h_index_date = h_index_dates[-1] if len(h_index_dates) else "unknown date"

page_specs = [
    {
        "page_title": "All researchers",
        "sample_key": "all_retained_event_units",
        "selector": lambda frame: frame,
    },
    {
        "page_title": "No earlier service",
        "sample_key": "no_earlier_broad_service_evidence",
        "selector": lambda frame: frame.loc[broad_first_service_mask(frame)],
    },
]

summary_records = []

with PdfPages(COMPARISON_FIGURE_OUT) as pdf:
    for page_idx, page in enumerate(page_specs):
        fig, axes = plt.subplots(
            nrows=len(plot_groups),
            ncols=4,
            figsize=(15.2, 8.6),
            gridspec_kw={"width_ratios": [2.25, 1.0, 1.0, 1.08]},
        )
        fig.suptitle(page["page_title"], fontsize=13.5, y=0.985)

        for row_idx, plot_group in enumerate(plot_groups):
            group_rows = event_rows.loc[event_rows["plot_group"].eq(plot_group)].copy()
            sample_rows = page["selector"](group_rows).copy()
            sample_units = event_units(sample_rows)
            n_units = sample_units["event_unit_id"].nunique()
            color = CONFERENCE_COLORS[plot_group]

            draw_event_axis(
                axes[row_idx, 0],
                sample_rows,
                color,
                f"{plot_group} -- N={n_units}",
                show_ylabel=True,
            )
            draw_hist_axis(
                axes[row_idx, 1],
                sample_units["career_age_at_first_pc"],
                bins=career_age_bins,
                color=color,
                title="Career age at first PC",
                xlabel="Years",
                xlim=(0, career_age_upper),
            )
            draw_hist_axis(
                axes[row_idx, 2],
                sample_units["h_index"],
                bins=h_index_bins,
                color=color,
                title=f"h-index\nOpenAlex {h_index_date}",
                xlabel="h-index",
                xlim=(0, h_index_upper),
            )
            draw_pre_pc_axis(axes[row_idx, 3], sample_rows, color)

            for ax in axes[row_idx, :]:
                ax.title.set_fontsize(8.2)
                ax.xaxis.label.set_size(7.8)
                ax.yaxis.label.set_size(7.8)
                ax.tick_params(axis="both", labelsize=7.3)

            summary_records.append(
                {
                    "page": page["page_title"],
                    "sample_key": page["sample_key"],
                    "plot_group": plot_group,
                    "n_event_units": int(n_units),
                    "median_career_age": sample_units["career_age_at_first_pc"].median(),
                    "mean_career_age": sample_units["career_age_at_first_pc"].mean(),
                    "median_h_index": sample_units["h_index"].median(),
                    "mean_h_index": sample_units["h_index"].mean(),
                }
            )

        handles = [
            plt.Line2D([0], [0], color="black", lw=0.8, ls="--"),
            plt.Line2D(
                [0],
                [0],
                color="black",
                lw=0.9,
                marker="_",
                markersize=8,
                linestyle="none",
            ),
        ]
        labels = ["PC-service year", r"95\% bootstrap CI"]
        axes[0, 0].legend(handles, labels, loc="lower left", frameon=False, fontsize=7.1)

        fig.tight_layout(rect=[0, 0, 1, 0.955], h_pad=2.0, w_pad=1.65)
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

summary = pd.DataFrame(summary_records)
summary.to_csv(SUMMARY_OUT, index=False)
REPORT_COMPARISON_FIGURE_OUT.write_bytes(COMPARISON_FIGURE_OUT.read_bytes())

print(COMPARISON_FIGURE_OUT.relative_to(PROJECT))
print(REPORT_COMPARISON_FIGURE_OUT.relative_to(PROJECT))
print(SUMMARY_OUT.relative_to(PROJECT))
display(summary)

## 6. Figure preview

In [ ]:
notebook_root = Path("..") / ".."
rel_path = COMPARISON_FIGURE_OUT.relative_to(PROJECT)
display(Markdown(f"`{rel_path}`"))
display(IFrame(src=str(notebook_root / rel_path), width="100%", height=760))